In [1]:
import os, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision import datasets
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, f1_score
from sklearn.preprocessing import label_binarize
from PIL import Image

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
from google.colab import drive
drive.mount('/content/drive')
print("Dataset For Model is successfully uploaded")

MODEL_PATH   = '/content/drive/MyDrive/dfu_wagner_v4.pth'
DATASET_ROOT = '/content/drive/MyDrive/Wagnar'
CLASS_NAMES  = ['Grade 1', 'Grade 2', 'Grade 3', 'Grade 4']
NUM_CLASSES  = 4
IMG_SIZE     = 224
DROPOUT      = 0.45
MEAN         = [0.485, 0.456, 0.406]
STD          = [0.229, 0.224, 0.225]



Device: cuda
Mounted at /content/drive
Dataset For Model is successfully uploaded


In [ ]:
def build_model():
    backbone = timm.create_model(
        'efficientnet_b0', pretrained=False,
        num_classes=0, drop_rate=DROPOUT, drop_path_rate=0.25,
    )
    in_features = backbone.num_features  # 1280

    class EfficientDFU(nn.Module):
        def __init__(self, bb):
            super().__init__()
            self.backbone = bb
            self.pool     = nn.AdaptiveAvgPool2d(1)
            self.head     = nn.Sequential(
                nn.Flatten(),
                nn.BatchNorm1d(in_features),
                nn.Dropout(p=DROPOUT),
                nn.Linear(in_features, 512),
                nn.SiLU(),
                nn.BatchNorm1d(512),
                nn.Dropout(p=DROPOUT * 0.5),
                nn.Linear(512, NUM_CLASSES),
            )
        def forward(self, x):
            x = self.backbone.forward_features(x)
            x = self.pool(x)
            return self.head(x)

    return EfficientDFU(backbone)


model      = build_model().to(DEVICE)
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
state_dict = checkpoint.get('model_state_dict', checkpoint)
state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)
model.eval()
print("Model loaded ✔")


tta_list = [
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.HorizontalFlip(p=1), A.Normalize(MEAN, STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.VerticalFlip(p=1),   A.Normalize(MEAN, STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE+24, IMG_SIZE+24), A.CenterCrop(IMG_SIZE, IMG_SIZE), A.Normalize(MEAN, STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.RandomRotate90(p=1), A.Normalize(MEAN, STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Transpose(p=1),      A.Normalize(MEAN, STD), ToTensorV2()]),
    A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.CLAHE(clip_limit=3, p=1), A.Normalize(MEAN, STD), ToTensorV2()]),
]


def evaluate_tta(split='test'):
    base = datasets.ImageFolder(os.path.join(DATASET_ROOT, split))
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    print(f"Running TTA (7 passes) on {len(base)} images...")
    for path, label in base.samples:
        img_np     = np.array(Image.open(path).convert('RGB'))
        probs_list = []
        for aug in tta_list:
            t = aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                probs_list.append(torch.softmax(model(t), dim=1).cpu().numpy()[0])
        avg = np.mean(probs_list, axis=0)
        all_probs.append(avg)
        all_preds.append(avg.argmax())
        all_labels.append(label)
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

preds, labels, all_probs = evaluate_tta('test')
acc = (preds == labels).mean()
print(f"\n{'='*50}")
print(f"  Test Accuracy (TTA×7) : {acc:.2%}")
print(f"{'='*50}\n")
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

def apply_correction(probs, penalty):
    p = probs.copy()
    p[:, 0] *= penalty
    return p / p.sum(axis=1, keepdims=True)

best_f1, best_penalty = 0, 1.0
print(f"{'Penalty':>8} | {'Accuracy':>9} | {'Macro F1':>9} | {'G2 Recall':>10} | {'G3 Recall':>10}")
print("─" * 58)
for penalty in [1.0, 0.95, 0.90, 0.85, 0.80, 0.75, 0.70, 0.65, 0.60]:
    adj   = apply_correction(all_probs, penalty)
    ap    = adj.argmax(axis=1)
    acc_p = (ap == labels).mean()
    mf1   = f1_score(labels, ap, average='macro')
    g2r   = (ap[labels==1] == 1).mean()
    g3r   = (ap[labels==2] == 2).mean()
    mark  = ' ← best' if mf1 > best_f1 else ''
    print(f"{penalty:>8.2f} | {acc_p:>8.2%} | {mf1:>9.4f} | {g2r:>9.2%} | {g3r:>9.2%}{mark}")
    if mf1 > best_f1:
        best_f1, best_penalty = mf1, penalty

print(f"\nBest penalty : {best_penalty}")

final_probs = apply_correction(all_probs, best_penalty)
final_preds = final_probs.argmax(axis=1)
final_acc   = (final_preds == labels).mean()
print(f"\nFinal Accuracy (after correction) : {final_acc:.2%}\n")
print(classification_report(labels, final_preds, target_names=CLASS_NAMES, digits=4))

Model loaded ✔
Running TTA (7 passes) on 1558 images...


In [ ]:
def plot_cm(y_true, y_pred):
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, fmt, title in zip(axes, [cm, cm_norm], ['d', '.1%'],
        ['Confusion Matrix (Counts)', 'Confusion Matrix (Normalised)']):
        sns.heatmap(data, annot=True, fmt=fmt, cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    linewidths=0.5, ax=ax, annot_kws={'size': 11})
        ax.set_xlabel('Predicted', fontweight='bold')
        ax.set_ylabel('Actual',    fontweight='bold')
        ax.set_title(title, fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

plot_cm(labels, final_preds)

In [ ]:
def plot_per_class(y_true, y_pred):
    per    = [(y_pred[y_true==i]==i).mean() for i in range(NUM_CLASSES)]
    colors = ['#2ecc71' if a>=0.65 else '#e67e22' if a>=0.5 else '#e74c3c' for a in per]
    plt.figure(figsize=(8, 4))
    bars = plt.bar(CLASS_NAMES, per, color=colors, edgecolor='black', width=0.5)
    for bar, a in zip(bars, per):
        plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f'{a:.1%}', ha='center', fontweight='bold')
    plt.ylim(0, 1.15)
    plt.axhline(0.65, color='green', linestyle='--', alpha=0.5, label='65% target')
    plt.title('Per-Class Accuracy (after correction)', fontweight='bold')
    plt.ylabel('Accuracy'); plt.legend(); plt.tight_layout(); plt.show()

plot_per_class(labels, final_preds)


In [ ]:
def plot_roc(y_true, y_prob):
    y_bin  = label_binarize(y_true, classes=list(range(NUM_CLASSES)))
    colors = ['#e74c3c','#e67e22','#f1c40f','#3498db']
    plt.figure(figsize=(8, 6))
    for i, (name, c) in enumerate(zip(CLASS_NAMES, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:,i], y_prob[:,i])
        plt.plot(fpr, tpr, color=c, lw=2.5, label=f'{name} (AUC={auc(fpr,tpr):.3f})')
    plt.plot([0,1],[0,1],'k--',lw=1.5,label='Random')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title('ROC Curves — Wagner Grade Classification', fontweight='bold')
    plt.legend(loc='lower right'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

plot_roc(labels, final_probs)



In [ ]:
WAGNER_INFO = {
    'Grade 1': 'Superficial ulcer — skin only, no infection.',
    'Grade 2': 'Deep ulcer → tendon / joint capsule / bone.',
    'Grade 3': 'Deep ulcer + abscess / osteomyelitis.',
    'Grade 4': 'Localised gangrene — forefoot or heel.',
}

def predict_image(image_path, penalty=best_penalty):
    img_np     = np.array(Image.open(image_path).convert('RGB'))
    probs_list = []
    model.eval()
    for aug in tta_list:
        t = aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs_list.append(torch.softmax(model(t), dim=1).cpu().numpy()[0])
    probs = np.mean(probs_list, axis=0)
    # Apply correction
    probs[0] *= penalty
    probs = probs / probs.sum()
    pred  = probs.argmax()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].imshow(Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE)))
    axes[0].axis('off'); axes[0].set_title('Input Image', fontweight='bold')
    colors = ['#3498db' if i != pred else '#e74c3c' for i in range(NUM_CLASSES)]
    bars = axes[1].barh(CLASS_NAMES, probs, color=colors, edgecolor='black')
    for bar, p in zip(bars, probs):
        axes[1].text(bar.get_width()+0.005, bar.get_y()+bar.get_height()/2,
                     f'{p:.2%}', va='center', fontweight='bold')
    axes[1].set_xlim(0, 1.15)
    axes[1].set_title('Wagner Grade Probabilities', fontweight='bold')
    plt.suptitle(
        f"Prediction: {CLASS_NAMES[pred]}  ({probs[pred]:.2%})\n"
        f"{WAGNER_INFO[CLASS_NAMES[pred]]}",
        fontsize=12, fontweight='bold', color='#c0392b'
    )
    plt.tight_layout(); plt.show()
    return CLASS_NAMES[pred], probs

In [ ]:
# ── Extend penalty search below 0.60 ─────────────────────────
best_f1, best_penalty = 0, 1.0

print(f"{'Penalty':>8} | {'Accuracy':>9} | {'Macro F1':>9} | {'G2 Recall':>10} | {'G3 Recall':>10}")
print("─" * 58)

for penalty in [0.60, 0.55, 0.50, 0.45, 0.40, 0.35, 0.30]:
    adj   = apply_correction(all_probs, penalty)
    ap    = adj.argmax(axis=1)
    acc_p = (ap == labels).mean()
    mf1   = f1_score(labels, ap, average='macro')
    g2r   = (ap[labels==1] == 1).mean()
    g3r   = (ap[labels==2] == 2).mean()
    g1r   = (ap[labels==0] == 0).mean()
    g4r   = (ap[labels==3] == 3).mean()
    mark  = ' ← best' if mf1 > best_f1 else ''
    print(f"{penalty:>8.2f} | {acc_p:>8.2%} | {mf1:>9.4f} | "
          f"{g2r:>9.2%} | {g3r:>9.2%}{mark}")
    if mf1 > best_f1:
        best_f1, best_penalty = mf1, penalty

print(f"\nBest penalty : {best_penalty}  (Macro F1: {best_f1:.4f})")

# Apply and show final
final_probs = apply_correction(all_probs, best_penalty)
final_preds = final_probs.argmax(axis=1)
print(f"\nFinal Accuracy : {(final_preds==labels).mean():.2%}")
print(classification_report(labels, final_preds, target_names=CLASS_NAMES, digits=4))
plot_cm(labels, final_preds)
plot_per_class(labels, final_preds)

In [ ]:
import torch

SAVE_PATH = '/content/drive/MyDrive/dfu_wagner_final_v5.pth'

torch.save({
    'model_state_dict': model.state_dict(),
    'class_names': CLASS_NAMES,
    'img_size': IMG_SIZE,
    'num_classes': NUM_CLASSES,
    'dropout': DROPOUT,
    'mean': MEAN,
    'std': STD,
    'best_penalty': best_penalty,
    'final_accuracy': float((final_preds == labels).mean()),
    'history': checkpoint.get('history', None),
    'best_val_acc': checkpoint.get('best_val_acc', None)
}, SAVE_PATH)

print(f"Model saved to: {SAVE_PATH} ✔")

**TESTING**

In [ ]:
# ── To predict a single image, uncomment and run: ────────────
predict_image('/content/drive/MyDrive/Wagnar/test/Grade 4/102_jpg.rf.0c645f77702740ba405071855d9548bd.jpg')

In [ ]:
# ── To predict a single image, uncomment and run: ────────────
predict_image('/content/drive/MyDrive/Wagnar/test/Grade 3/-a-e-u-a_jpg.rf.a9ff6552b43699128daf8a037df51c11.jpg')

In [ ]:
# ── To predict a single image, uncomment and run: ────────────
predict_image('/content/drive/MyDrive/Wagnar/test/Grade 2/13_jpg.rf.4a8553aa35e8fc197c2cfa90608c4aed.jpg')

In [ ]:
# ── To predict a single image, uncomment and run: ────────────
predict_image('/content/drive/MyDrive/Wagnar/test/Grade 1/154_jpg.rf.ec330734efd0f5e818d42974d63301d2.jpg')